# 🚀 Tái Hiện Thuật Toán LiDAR (Lookahead Sample Reward Guidance) - Bảng 2
### **Bài báo**: [Lookahead Sample Reward Guidance for Test-Time Scaling of Diffusion Models (ICML 2026 Spotlight)](https://arxiv.org/abs/2602.03211)
### **Mục tiêu**: Chạy lại mã nguồn và tái hiện kết quả của **Bảng 2**: **SD v1.5 + LiDAR (DPM-5 / $n=50$)** trên tập prompt GenEval.

---
### 📊 Kết quả mục tiêu trong bài báo (Bảng 2):
| Mô hình Backbone | Phương pháp Sampling | ImageReward (↑) | CLIP Score (↑) | HPS v2.1 (↑) | GenEval (↑) | Thời gian (s/lần) | VRAM (GiB) |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **SD v1.5 (DDPM 100 bước)** | **LiDAR (DPM-5 / $n=50$)** | **0.384** | **0.278** | **0.276** | **0.478** | 13.41s | 8.90 GiB |
| **SD v1.5 (DDIM 50 bước)** | **LiDAR (DPM-5 / $n=50$)** | **0.378** | **0.278** | **0.277** | **0.475** | 9.92s | 8.90 GiB |
| *Vanilla SD v1.5 (DDPM 100)* | Baseline gốc | 0.001 | 0.271 | 0.263 | 0.426 | 7.07s | 8.90 GiB |
| *Vanilla SD v1.5 (DDIM 50)* | Baseline gốc | -0.125 | 0.269 | 0.270 | 0.423 | 3.58s | 8.90 GiB |

---
### 🛠 Cơ Chế Lưu Trữ & Chạy Tiếp Tục (Resume Checkpoint):
1. **Lưu tự động sau mỗi Prompt**: Mỗi prompt khi sinh xong sẽ lập tức lưu latent `latent.pt` và chỉ số `results.json` vào thư mục riêng (`00000/`, `00001/`,...).
2. **Tự động bỏ qua prompt đã hoàn thành**: Nếu phiên làm việc bị ngắt kết nối hoặc bạn tắt đi bật lại, mã nguồn có cờ `--resume` sẽ quét thư mục, tự động nạp kết quả cũ và **chỉ chạy tiếp các prompt còn lại**.
3. **Tự động khôi phục từ Save Version cũ**: Nếu phiên trước chạy **Save Version (Save & Run All)** bị hết giờ, bạn chỉ cần Add Output phiên đó vào Input. Notebook sẽ **tự động quét và khôi phục toàn bộ prompt đã sinh** để chạy nối tiếp!


## 1. Kiểm tra Môi trường Hệ thống & GPU


In [ ]:
import os, sys, torch

print(f"Phiên bản Python: {sys.version}")
print(f"Phiên bản PyTorch: {torch.__version__}")
print(f"Hỗ trợ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"Số lượng GPU khả dụng: {n_gpus}")
    for g_i in range(n_gpus):
        print(f" - GPU {g_i}: {torch.cuda.get_device_name(g_i)} ({torch.cuda.get_device_properties(g_i).total_memory / (1024**3):.2f} GB VRAM)")
!nvidia-smi


## 2. Thiết lập Mã Nguồn, Đồng Bộ Checkpoint Cũ & Cài Đặt Thư Viện


In [ ]:
# Thiết lập thư mục làm việc trên Kaggle
import os, shutil, glob, subprocess, threading

REPO_DIR = "/kaggle/working/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

if os.path.exists(f"{REPO_DIR}/Diffusion-LiDAR-Sampling"):
    WORKDIR = f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
else:
    WORKDIR = REPO_DIR

os.chdir(WORKDIR)
%cd {WORKDIR}
print("Thư mục làm việc hiện tại:", os.getcwd())

# Tạo sẵn các thư mục đầu ra
os.makedirs(f"{WORKDIR}/Lookahead_samples", exist_ok=True)
os.makedirs(f"{WORKDIR}/Target_samples", exist_ok=True)

# ==================== TỰ ĐỘNG KHÔI PHỤC DỮ LIỆU TỪ SAVE VERSION / ZIP CŨ ====================
# 1. Quét và giải nén tất cả file .zip có trong /kaggle/input hoặc /kaggle/working
zip_files = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("/kaggle/working/*.zip")
for zf in zip_files:
    print(f"📦 Tìm thấy file zip dữ liệu: {zf}. Đang giải nén...")
    try:
        shutil.unpack_archive(zf, WORKDIR)
    except Exception as e:
        print(f"⚠️ Lỗi giải nén: {e}")

# 2. Tự động đồng bộ toàn bộ thư mục Lookahead_samples từ Output của phiên Save Version trước (nếu có)
for l_dir in glob.glob("/kaggle/input/**/Lookahead_samples/*", recursive=True):
    if os.path.isdir(l_dir):
        base_name = os.path.basename(l_dir)
        dest_dir = os.path.join(WORKDIR, "Lookahead_samples", base_name)
        os.makedirs(dest_dir, exist_ok=True)
        for p_dir in glob.glob(os.path.join(l_dir, "[0-9]*")):
            p_name = os.path.basename(p_dir)
            p_dest = os.path.join(dest_dir, p_name)
            if not os.path.exists(p_dest) and os.path.isdir(p_dir):
                shutil.copytree(p_dir, p_dest)

# 3. Tự động đồng bộ toàn bộ thư mục Target_samples từ Output của phiên Save Version trước (nếu có)
for t_dir in glob.glob("/kaggle/input/**/Target_samples/*", recursive=True):
    if os.path.isdir(t_dir):
        base_name = os.path.basename(t_dir)
        dest_dir = os.path.join(WORKDIR, "Target_samples", base_name)
        os.makedirs(dest_dir, exist_ok=True)
        for p_dir in glob.glob(os.path.join(t_dir, "[0-9]*")):
            p_name = os.path.basename(p_dir)
            p_dest = os.path.join(dest_dir, p_name)
            if not os.path.exists(p_dest) and os.path.isdir(p_dir):
                shutil.copytree(p_dir, p_dest)

n_look = len(glob.glob(f"{WORKDIR}/Lookahead_samples/*/[0-9]*"))
n_targ = len(glob.glob(f"{WORKDIR}/Target_samples/*/[0-9]*"))
print(f"✅ Đã đồng bộ xong dữ liệu: {n_look} Lookahead prompts, {n_targ} Target prompts sẵn sàng!")

# ==================== HÀM TIỆN ÍCH CHẠY 2 GPU HIỂN THỊ LOG TRỰC TIẾP ====================
def run_commands_parallel(cmd0, cmd1):
    p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    def stream_logs(proc, prefix):
        for line in iter(proc.stdout.readline, ''):
            if line.strip():
                print(f"{prefix} {line.strip()}")
        proc.stdout.close()
    t0 = threading.Thread(target=stream_logs, args=(p0, "[GPU 0]"))
    t1 = threading.Thread(target=stream_logs, args=(p1, "[GPU 1]"))
    t0.start(); t1.start()
    t0.join(); t1.join()
    p0.wait(); p1.wait()

# ==================== CÀI ĐẶT THƯ VIỆN ====================
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas

# Tải file vocab và trọng số cho hpsv2
import urllib.request, hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)

# Tải trước mô hình HPSv2.1 tránh xung đột khi chạy 2 GPU song song
from huggingface_hub import hf_hub_download
hps_cache = os.path.expanduser("~/.cache/hpsv2")
os.makedirs(hps_cache, exist_ok=True)
hps_ckpt = os.path.join(hps_cache, "HPS_v2.1_compressed.pt")
if not os.path.exists(hps_ckpt) or os.path.getsize(hps_ckpt) < 1000000:
    print("⏳ Đang tải trước trọng số HPSv2.1...")
    try:
        hf_hub_download(repo_id="xswu/HPSv2", filename="HPS_v2.1_compressed.pt", local_dir=hps_cache)
        print("✅ Đã tải xong HPSv2.1!")
    except Exception:
        try:
            urllib.request.urlretrieve("https://huggingface.co/xswu/HPSv2/resolve/main/HPS_v2.1_compressed.pt", hps_ckpt)
            print("✅ Đã tải xong HPSv2.1!")
        except Exception as e:
            print(f"Lưu ý tải HPS: {e}")

print("✅ Môi trường trên Kaggle đã được cài đặt hoàn tất!")


## 3. Cấu hình Siêu Tham Số Thí Nghiệm (Thiết lập theo Bảng 2)


In [ ]:
# ==================== CẤU HÌNH SIÊU THAM SỐ ====================
NUM_GPUS = 2                     # Đặt = 2 nếu bật 2x GPU T4 trên Kaggle (chạy song song), hoặc = 1 nếu chỉ dùng 1 GPU
SEED = 100                       # Random seed (100 hoặc 42)
NUM_LOOKAHEAD_PARTICLES = 50     # Số hạt lookahead n = 50
LOOKAHEAD_STEPS = 5              # Số bước DPM-Solver = 5 (DPM-5)
LOOKAHEAD_TAG = f"{SEED}_{NUM_LOOKAHEAD_PARTICLES}_{LOOKAHEAD_STEPS}"
SKIP_PHASE_1 = False             # Dat = True de bo qua Phase 1 neu da co san kho Lookahead (hoac code se tu dong phat hien)

# Tham số lấy mẫu đích Phase 2 (SD v1.5 với DDIM 50 bước - siêu tốc ~3 giờ)
MODEL_NAME = "runwayml/stable-diffusion-v1-5"
NUM_TARGET_STEPS = 50            # 50 bước DDIM (theo Bảng 2)
ETA = 0.0                        # eta = 0.0 cho DDIM (hoặc 1.0 cho DDPM)
TARGET_PARTICLES = 4             # 4 ảnh trên mỗi prompt (chuẩn đánh giá GenEval)
SCALE = 15.0                     # He so guidance s = 15.0 de dat ImageReward dinh 0.378 theo Bang 2
LAMBDA = 5000                    # Hệ số nhiệt độ lambda = 5000
RESAMPLE_T_END = 200             # Ngưỡng kết thúc guidance sớm [1.0, 0.2]
TOP_K = 50                       # Chọn top-k lookaheads (50)

# Dữ liệu Prompt và Giới hạn số lượng (thử nghiệm: 10, toàn bộ: 553)
PROMPT_FILE = "prompt_files/geneval_metadata.jsonl"
MAX_PROMPTS = 553

RUN_NAME = f"LiDAR_SD15_DPM5_n50_DDIM50_scale{SCALE}_seed{SEED}"

print(f"Cấu hình Số GPU: {NUM_GPUS} GPU")
print(f"Tên lượt chạy: {RUN_NAME}")
print(f"Cấu hình Target: DDIM {NUM_TARGET_STEPS} bước (eta={ETA})")
print(f"Đường dẫn Lookahead: {LOOKAHEAD_TAG}")
print(f"Tổng số Prompt cần xử lý: {MAX_PROMPTS}")


## 4. Giai Đoạn 1 (Phase 1): Lấy Mẫu Lookahead & Đánh Giá Reward


In [ ]:
# Thuc thi Giai doan 1: Lookahead Sampling
import os, glob
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.chdir(WORKDIR)

existing_lookaheads = len(glob.glob(f"{WORKDIR}/Lookahead_samples/{LOOKAHEAD_TAG}/[0-9]*"))
if SKIP_PHASE_1 or existing_lookaheads >= MAX_PROMPTS:
    print(f"⏩ [SKIP PHASE 1] Da co san {existing_lookaheads}/{MAX_PROMPTS} prompt Lookahead! Bo qua Phase 1, chuyen thang sang Phase 2.")
else:
    if NUM_GPUS == 2:
        print("🚀 [2 GPU] Dang chay song song Phase 1 tren GPU 0 va GPU 1 (moi GPU 1/2 so prompt)...")
        cmd0 = f"""python lookahead_sampling.py \
            --seed={SEED} --num_particles={NUM_LOOKAHEAD_PARTICLES} --num_inference_steps={LOOKAHEAD_STEPS} \
            --model_name="{MODEL_NAME}" --guidance_reward_fn="ImageReward" --metrics_to_compute="ImageReward#Clip-Score" \
            --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --gpu_id=0 --num_shards=2 --shard_id=0 --resume"""
        cmd1 = f"""python lookahead_sampling.py \
            --seed={SEED} --num_particles={NUM_LOOKAHEAD_PARTICLES} --num_inference_steps={LOOKAHEAD_STEPS} \
            --model_name="{MODEL_NAME}" --guidance_reward_fn="ImageReward" --metrics_to_compute="ImageReward#Clip-Score" \
            --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --gpu_id=1 --num_shards=2 --shard_id=1 --resume"""
        run_commands_parallel(cmd0, cmd1)
        print("✅ Ca 2 GPU da hoan thanh Giai doan 1!")
    else:
        print("🚀 [1 GPU] Dang chay Phase 1 tren 1 GPU...")
        !python lookahead_sampling.py \
            --seed={SEED} --num_particles={NUM_LOOKAHEAD_PARTICLES} --num_inference_steps={LOOKAHEAD_STEPS} \
            --model_name="{MODEL_NAME}" --guidance_reward_fn="ImageReward" --metrics_to_compute="ImageReward#Clip-Score" \
            --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --resume


## 5. Giai Đoạn 2 (Phase 2): Lấy Mẫu Đích LiDAR Sampling (DDIM 50 bước)


In [ ]:
# Thuc thi Giai doan 2: LiDAR Steering Sampling
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.chdir(WORKDIR)

if NUM_GPUS == 2:
    print("🚀 [2 GPU] Dang chay song song Phase 2 tren GPU 0 va GPU 1 (moi GPU 1/2 so prompt)...")
    cmd0 = f"""python LiDAR_sampling.py \
        --seed={SEED} --model_name="{MODEL_NAME}" --num_particles={TARGET_PARTICLES} --num_inference_steps={NUM_TARGET_STEPS} \
        --eta={ETA} --use_rag --lookahead_path="{LOOKAHEAD_TAG}" --top_k={TOP_K} --scale={SCALE} --lmbda={LAMBDA} --resample_t_end={RESAMPLE_T_END} \
        --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --guidance_reward_fn="ImageReward" \
        --metrics_to_compute="ImageReward#Clip-Score#Clip-Diversity#HumanPreference#AS" \
        --save_individual_images --run_name="{RUN_NAME}" --gpu_id=0 --num_shards=2 --shard_id=0 --resume"""
    cmd1 = f"""python LiDAR_sampling.py \
        --seed={SEED} --model_name="{MODEL_NAME}" --num_particles={TARGET_PARTICLES} --num_inference_steps={NUM_TARGET_STEPS} \
        --eta={ETA} --use_rag --lookahead_path="{LOOKAHEAD_TAG}" --top_k={TOP_K} --scale={SCALE} --lmbda={LAMBDA} --resample_t_end={RESAMPLE_T_END} \
        --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --guidance_reward_fn="ImageReward" \
        --metrics_to_compute="ImageReward#Clip-Score#Clip-Diversity#HumanPreference#AS" \
        --save_individual_images --run_name="{RUN_NAME}" --gpu_id=1 --num_shards=2 --shard_id=1 --resume"""
    run_commands_parallel(cmd0, cmd1)
    print("✅ Ca 2 GPU da hoan thanh Giai doan 2!")
else:
    print("🚀 [1 GPU] Dang chay Phase 2 tren 1 GPU...")
    !python LiDAR_sampling.py \
        --seed={SEED} --model_name="{MODEL_NAME}" --num_particles={TARGET_PARTICLES} --num_inference_steps={NUM_TARGET_STEPS} \
        --eta={ETA} --use_rag --lookahead_path="{LOOKAHEAD_TAG}" --top_k={TOP_K} --scale={SCALE} --lmbda={LAMBDA} --resample_t_end={RESAMPLE_T_END} \
        --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --guidance_reward_fn="ImageReward" \
        --metrics_to_compute="ImageReward#Clip-Score#Clip-Diversity#HumanPreference#AS" \
        --save_individual_images --run_name="{RUN_NAME}" --resume


## 6. Đánh Giá Tự Động GenEval Benchmark (Hugging Face & Torchvision Native)
Tự động chấm điểm độ chuẩn xác của các ảnh đã sinh theo 6 tiêu chí của GenEval (vật thể đơn, 2 vật thể, đếm số lượng, màu sắc, vị trí tương đối, gán màu) bằng Torchvision & HuggingFace.


In [ ]:
# ==============================================================================
# 🎯 GENEVAL BENCHMARK EVALUATION (100% NATIVE HUGGINGFACE & PYTORCH)
# ==============================================================================
import os, glob, json, urllib.request
import numpy as np
import pandas as pd
from PIL import Image
import torch
from tqdm import tqdm

from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as TF
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Thiết bị tính toán GenEval: {device}")

# 1. Tìm hoặc tải file metadata chuẩn của GenEval
metadata_candidates = glob.glob("/kaggle/**/geneval_metadata.jsonl", recursive=True) + glob.glob("**/geneval_metadata.jsonl", recursive=True)
if metadata_candidates and os.path.exists(metadata_candidates[0]):
    metadata_path = metadata_candidates[0]
else:
    metadata_path = "/kaggle/working/geneval_metadata.jsonl"
    url = "https://raw.githubusercontent.com/leekwanreal/RS-LiDAR/main/prompt_files/geneval_metadata.jsonl"
    urllib.request.urlretrieve(url, metadata_path)

prompts_meta = []
with open(metadata_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            prompts_meta.append(json.loads(line.strip()))
print(f"📋 Đã nạp {len(prompts_meta)} prompt tiêu chuẩn GenEval.")

# 2. Quét tất cả ảnh đã sinh từ Phase 2
target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
prompt_dir_map = {}
for p_dir in glob.glob(f"{target_dir}/[0-9]*"):
    try:
        p_idx = int(os.path.basename(p_dir))
    except ValueError:
        continue
    imgs = sorted(glob.glob(f"{p_dir}/samples/*.png"))
    if not imgs:
        imgs = sorted([img for img in glob.glob(f"{p_dir}/*.png") if not img.endswith("grid.png")])
    if imgs:
        prompt_dir_map[p_idx] = imgs

print(f"🎯 Đã tìm thấy {len(prompt_dir_map)} prompts có ảnh để chấm điểm.")

overall_geneval = None
if prompt_dir_map:
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    detector = fasterrcnn_resnet50_fpn(weights=weights).to(device).eval()
    coco_classes = weights.meta["categories"]
    
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    COLORS = ["red", "orange", "yellow", "green", "blue", "purple", "pink", "brown", "black", "white"]
    color_prompts = [f"a photo of a {c} object" for c in COLORS]
    
    def classify_crop_color(crop_img):
        inputs = clip_processor(text=color_prompts, images=crop_img, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            outputs = clip_model(**inputs)
            best_idx = outputs.logits_per_image.argmax(dim=-1).item()
            return COLORS[best_idx]
    
    task_results = {"single_object": [], "two_object": [], "counting": [], "colors": [], "position": [], "color_attr": []}
    for p_idx in tqdm(sorted(prompt_dir_map.keys()), desc="Chấm điểm GenEval"):
        if p_idx >= len(prompts_meta):
            continue
        meta = prompts_meta[p_idx]
        tag = meta.get("tag", "single_object")
        if tag not in task_results:
            continue
        
        images_paths = prompt_dir_map[p_idx]
        prompt_scores = []
        for img_path in images_paths:
            try:
                img = Image.open(img_path).convert("RGB")
                img_t = TF.to_tensor(img).to(device)
                with torch.no_grad():
                    preds = detector([img_t])[0]
                
                scores = preds["scores"].cpu().numpy()
                labels = preds["labels"].cpu().numpy()
                boxes = preds["boxes"].cpu().numpy()
                keep = scores > 0.35
                detected_labels, detected_boxes = labels[keep], boxes[keep]
                
                detected_objects = []
                for lbl, box in zip(detected_labels, detected_boxes):
                    c_name = coco_classes[lbl].lower()
                    x1, y1, x2, y2 = box
                    if (x2 - x1 > 12 and y2 - y1 > 12):
                        crop = img.crop((max(0, x1), max(0, y1), min(img.width, x2), min(img.height, y2)))
                        pred_color = classify_crop_color(crop)
                    else:
                        pred_color = "unknown"
                    detected_objects.append({"class": c_name, "box": box, "center_x": (x1+x2)/2.0, "center_y": (y1+y2)/2.0, "color": pred_color})
                
                success = False
                includes = meta.get("include", [])
                if tag == "single_object":
                    req_cls = includes[0]["class"].lower()
                    success = any(req_cls in obj["class"] or obj["class"] in req_cls for obj in detected_objects)
                elif tag == "two_object":
                    req1, req2 = includes[0]["class"].lower(), includes[1]["class"].lower()
                    success = any(req1 in obj["class"] or obj["class"] in req1 for obj in detected_objects) and any(req2 in obj["class"] or obj["class"] in req2 for obj in detected_objects)
                elif tag == "counting":
                    req_cls, target_count = includes[0]["class"].lower(), includes[0]["count"]
                    found_count = sum(1 for obj in detected_objects if req_cls in obj["class"] or obj["class"] in req_cls)
                    success = (found_count == target_count)
                elif tag == "colors":
                    req_cls, req_color = includes[0]["class"].lower(), includes[0]["color"].lower()
                    success = any((req_cls in obj["class"] or obj["class"] in req_cls) and (obj["color"] == req_color) for obj in detected_objects)
                elif tag == "position":
                    req1, req2 = includes[0]["class"].lower(), includes[1]["class"].lower()
                    pos_type = includes[1].get("position", ["right of", 0])[0]
                    o1_list = [o for o in detected_objects if req1 in o["class"] or o["class"] in req1]
                    o2_list = [o for o in detected_objects if req2 in o["class"] or o["class"] in req2]
                    if o1_list and o2_list:
                        o1, o2 = o1_list[0], o2_list[0]
                        if "right" in pos_type: success = (o2["center_x"] > o1["center_x"])
                        elif "left" in pos_type: success = (o2["center_x"] < o1["center_x"])
                        elif "above" in pos_type or "top" in pos_type: success = (o2["center_y"] < o1["center_y"])
                        elif "below" in pos_type or "bottom" in pos_type: success = (o2["center_y"] > o1["center_y"])
                        else: success = True
                elif tag == "color_attr":
                    success = all(any((inc["class"].lower() in obj["class"] or obj["class"] in inc["class"].lower()) and (obj["color"] == inc["color"].lower()) for obj in detected_objects) for inc in includes)
                
                prompt_scores.append(1.0 if success else 0.0)
            except Exception as e:
                pass
            except Exception as e:
                pass
        
        if prompt_scores:
            task_results[tag].append(np.mean(prompt_scores))
    
    summary_rows = []
    all_means = []
    for t_name, scores in task_results.items():
        mean_val = np.mean(scores) if scores else 0.0
        if scores: all_means.append(mean_val)
        summary_rows.append({"Nhiệm Vụ (Task)": t_name, "Số Lượng Prompt": len(scores), "Độ Chính Xác (Accuracy ↑)": f"{mean_val:.4f}"})
    
    overall_geneval = np.mean(all_means) if all_means else 0.0
    summary_rows.append({"Nhiệm Vụ (Task)": "🔥 OVERALL GENEVAL BENCHMARK", "Số Lượng Prompt": sum(len(s) for s in task_results.values()), "Độ Chính Xác (Accuracy ↑)": f"{overall_geneval:.4f}"})
    df_geneval = pd.DataFrame(summary_rows)
    print("\n" + "="*70)
    print("📊 BẢNG TỔNG HỢP ĐIỂM SỐ GENEVAL BENCHMARK")
    print("="*70)
    print(df_geneval.to_string(index=False))
    print("="*70)
    df_geneval.to_csv(f"{target_dir}/geneval_summary.csv", index=False)
    df_geneval.to_csv("/kaggle/working/geneval_summary.csv", index=False)


## 7. Bảng Đối Chiếu Kết Quả Toàn Diện Với Bảng 2 (ICML 2026)


In [ ]:
import json, glob, os
import numpy as np
import pandas as pd
from IPython.display import display

target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
result_files = sorted(glob.glob(f"{target_dir}/[0-9]*/results.json"))

if result_files:
    print(f"📊 Đang tổng hợp chỉ số đánh giá từ {len(result_files)} prompt...")
    metric_keys = ["ImageReward", "Clip-Score", "HumanPreference", "Clip-Diversity", "AS"]
    collected_means = {k: [] for k in metric_keys}
    
    for rf in result_files:
        try:
            with open(rf, "r") as f:
                res = json.load(f)
            for k in metric_keys:
                if k in res and "mean" in res[k]:
                    collected_means[k].append(res[k]["mean"])
        except Exception:
            pass
    
    final_metrics = {}
    for k, vals in collected_means.items():
        if vals:
            final_metrics[k] = {
                "mean": float(np.mean(vals)),
                "std": float(np.std(vals)),
                "min": float(np.min(vals)),
                "max": float(np.max(vals)),
            }
    
    # Thêm GenEval vào final_metrics
    if 'overall_geneval' in locals() and overall_geneval is not None:
        final_metrics["GenEval"] = {"mean": float(overall_geneval)}
    
    with open(f"{target_dir}/final_metrics.json", "w") as f:
        json.dump(final_metrics, f, indent=4)
    
    ir_val = final_metrics.get('ImageReward', {}).get('mean', 0.0)
    clip_val = final_metrics.get('Clip-Score', {}).get('mean', 0.0)
    hps_val = final_metrics.get('HumanPreference', {}).get('mean', 0.0)
    div_val = final_metrics.get('Clip-Diversity', {}).get('mean', 0.0)
    as_val = final_metrics.get('AS', {}).get('mean', 0.0)
    ge_val = final_metrics.get('GenEval', {}).get('mean', None)
    ge_str = f"{ge_val:.4f}" if ge_val is not None else "Đang tính..."
    
    table_data = [
        {
            "Mô Hình / Phương Pháp": "SD v1.5 Gốc (Chưa lái)",
            "Số bước": "50 DDIM",
            "ImageReward ↑": "-0.125",
            "CLIP-Score ↑": "0.269",
            "HPS v2.1 ↑": "0.270",
            "GenEval ↑": "0.423",
            "Đánh Giá": "Baseline"
        },
        {
            "Mô Hình / Phương Pháp": "BÀI BÁO BẢNG 2 (LiDAR DDIM-50)",
            "Số bước": "50 DDIM",
            "ImageReward ↑": "0.378",
            "CLIP-Score ↑": "0.278",
            "HPS v2.1 ↑": "0.277",
            "GenEval ↑": "0.475",
            "Đánh Giá": "Target Benchmark"
        },
        {
            "Mô Hình / Phương Pháp": "BÀI BÁO BẢNG 2 (LiDAR DDPM-100)",
            "Số bước": "100 DDPM",
            "ImageReward ↑": "0.384",
            "CLIP-Score ↑": "0.278",
            "HPS v2.1 ↑": "0.276",
            "GenEval ↑": "0.478",
            "Đánh Giá": "Upper Bound"
        },
        {
            "Mô Hình / Phương Pháp": "🔥 KẾT QUẢ CHẠY THỰC TẾ (Ours)",
            "Số bước": f"{NUM_TARGET_STEPS} {'DDIM' if ETA==0.0 else 'DDPM'}",
            "ImageReward ↑": f"{ir_val:.4f}",
            "CLIP-Score ↑": f"{clip_val:.4f}",
            "HPS v2.1 ↑": f"{hps_val:.4f}",
            "GenEval ↑": ge_str,
            "Đánh Giá": f"Δ IR: {ir_val - 0.378:+.3f}"
        }
    ]
    
    df = pd.DataFrame(table_data)
    print("\n======================= 📊 BẢNG ĐỐI CHIẾU KẾT QUẢ VỚI BÀI BÁO (BẢNG 2) =======================")
    print(df.to_string(index=False))
    display(df)
    
    df.to_csv("/kaggle/working/table2_comparison.csv", index=False)
    df.to_csv(f"{target_dir}/table2_comparison.csv", index=False)
    with open("/kaggle/working/table2_comparison.md", "w", encoding="utf-8") as f:
        f.write(df.to_markdown(index=False))
    with open("/kaggle/working/final_metrics.json", "w", encoding="utf-8") as f:
        json.dump(final_metrics, f, indent=4)
    print("\n💾 Đã tự động xuất bảng kết quả đầy đủ (kèm GenEval) ra file:")
    print(" - CSV: /kaggle/working/table2_comparison.csv")
    print(" - Markdown: /kaggle/working/table2_comparison.md")
    print(" - JSON: /kaggle/working/final_metrics.json")


## 8. Trực Quan Hóa Các Ảnh Mẫu Đã Sinh


In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
grid_images = sorted(glob.glob(f"{target_dir}/*/grid.png"))

if grid_images:
    print(f"Tìm thấy {len(grid_images)} lưới ảnh prompt. Đang hiển thị tối đa 3 prompt đầu tiên:")
    for img_path in grid_images[:3]:
        img = Image.open(img_path)
        plt.figure(figsize=(16, 4))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Chỉ số Prompt: {os.path.basename(os.path.dirname(img_path))}")
        plt.show()
else:
    print("Chưa tìm thấy ảnh lưới mẫu nào.")
